In [ ]:
!pip install -q kagglehub

import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler

In [ ]:
path = kagglehub.dataset_download("anuchhetry/electronic-health-record")

print("Dataset downloaded successfully!")
print("Path:", path)

100%|██████████| 75.5k/75.5k [00:00<00:00, 445kB/s]

Extracting files...
Dataset downloaded successfully!
Path: /root/.cache/kagglehub/datasets/anuchhetry/electronic-health-record/versions/1


In [ ]:
csv_files = [file for file in os.listdir(path) if file.endswith('.csv')]

print("Files found:")
print(csv_files)

file_path = os.path.join(path, csv_files[0])

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Files found:
['EHR.csv']
Dataset loaded successfully!
Shape: (1447, 29)


In [ ]:
df.head()

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,admissionheight,hospitaladmittime24,...,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid
0,210014,182373,Male,45,Caucasian,73,89,"Hypertension, uncontrolled (for cerebrovascula...",178.0,13:08:59,...,Direct Admit,1,admit,116.0,112.7,15:00:00,4424,Skilled Nursing Facility,Alive,002-10665
1,200026,174624,Male,50,Caucasian,71,87,Ablation or mapping of cardiac conduction pathway,177.8,10:41:00,...,Operating Room,1,admit,106.1,106.1,17:40:00,1548,Home,Alive,002-10715
2,221131,190993,Male,83,Caucasian,71,87,"Endarterectomy, carotid",175.3,21:43:00,...,Operating Room,1,admit,NaN,72.1,17:46:00,1203,Home,Alive,002-10249
3,221215,191054,Male,49,Caucasian,71,87,"Infarction, acute myocardial (MI)",185.4,03:16:00,...,Emergency Department,1,admit,145.3,146.6,19:07:00,1562,Home,Alive,002-10627
4,217835,188445,Male,57,Caucasian,73,92,"CABG alone, coronary artery bypass grafting",172.7,01:09:00,...,Operating Room,1,admit,NaN,80.4,08:25:00,4719,Floor,Alive,002-10324


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1447 entries, 0 to 1446
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patientunitstayid          1447 non-null   int64  
 1   patienthealthsystemstayid  1447 non-null   int64  
 2   gender                     1444 non-null   object 
 3   age                        1446 non-null   object 
 4   ethnicity                  1405 non-null   object 
 5   hospitalid                 1447 non-null   int64  
 6   wardid                     1447 non-null   int64  
 7   apacheadmissiondx          1267 non-null   object 
 8   admissionheight            1402 non-null   float64
 9   hospitaladmittime24        1447 non-null   object 
 10  hospitaladmitoffset        1447 non-null   int64  
 11  hospitaladmitsource        1218 non-null   object 
 12  hospitaldischargeyear      1447 non-null   int64  
 13  hospitaldischargetime24    1447 non-null   objec

In [ ]:
missing_values = df.isnull().sum()

print("Missing values:")
print(missing_values[missing_values > 0].sort_values(ascending=False))

Missing values:
dischargeweight              576
hospitaladmitsource          229
apacheadmissiondx            180
admissionweight              134
admissionheight               45
ethnicity                     42
unitadmitsource               19
hospitaldischargelocation      8
hospitaldischargestatus        7
unitdischargelocation          5
gender                         3
unitdischargestatus            2
age                            1
dtype: int64


In [ ]:
import matplotlib.pyplot as plt

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 6))
missing.plot(kind='bar')

plt.title("Missing Values in EHR Dataset")
plt.xlabel("Features")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=90)

plt.show()

<Figure size 1200x600 with 1 Axes>

In [ ]:
for column in df.columns:

    if df[column].isnull().sum() > 0:

        if pd.api.types.is_numeric_dtype(df[column]):
            df[column] = df[column].fillna(df[column].median())

        else:
            df[column] = df[column].fillna(df[column].mode()[0])

print("Missing values handled successfully.")

Missing values handled successfully.


In [ ]:
print("Total missing values remaining:",
      df.isnull().sum().sum())

Total missing values remaining: 0


In [ ]:
print("Duplicate records before cleaning:",
      df.duplicated().sum())

Duplicate records before cleaning: 0


In [ ]:
df = df.drop_duplicates()

print("Duplicate records after cleaning:",
      df.duplicated().sum())

Duplicate records after cleaning: 0


In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

Q1 = df[numeric_columns].quantile(0.25)
Q3 = df[numeric_columns].quantile(0.75)

IQR = Q3 - Q1

outliers = (
    (df[numeric_columns] < (Q1 - 1.5 * IQR)) |
    (df[numeric_columns] > (Q3 + 1.5 * IQR))
)

print("Number of outliers detected:",
      outliers.sum().sum())

Number of outliers detected: 1016


In [ ]:
for column in numeric_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    df[column] = df[column].clip(
        lower_limit,
        upper_limit
    )

print("Outliers handled using IQR capping.")

Outliers handled using IQR capping.


In [ ]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

scaler = StandardScaler()

df[numeric_columns] = scaler.fit_transform(
    df[numeric_columns]
)

print("Numerical features standardized successfully.")

Numerical features standardized successfully.


In [ ]:
df[numeric_columns].head()

,patientunitstayid,patienthealthsystemstayid,hospitalid,wardid,admissionheight,hospitaladmitoffset,hospitaldischargeyear,hospitaldischargeoffset,unitvisitnumber,admissionweight,dischargeweight,unitdischargeoffset
0,-1.132469,-1.158683,-1.272693,-1.336145,0.925308,0.698450,0.731071,-0.500157,0.0,1.547460,1.731321,0.500001
1,-1.158384,-1.187406,-1.311036,-1.348736,0.906222,0.479388,-1.367857,-0.960001,0.0,1.096624,1.731321,-0.618168
2,-1.103624,-1.126731,-1.311036,-1.348736,0.667650,0.704234,-1.367857,-1.000170,0.0,-0.178469,-0.692601,-0.752301
3,-1.103406,-1.126505,-1.311036,-1.348736,1.631483,0.104886,-1.367857,-0.957409,0.0,2.514025,1.731321,-0.612725
4,-1.112176,-1.136176,-1.272693,-1.317259,0.419534,-1.404688,-1.367857,0.100605,0.0,-0.178469,-0.045390,0.614695


In [ ]:
categorical_columns = df.select_dtypes(
    include=['object', 'category']
).columns

print("Number of categorical columns:",
      len(categorical_columns))

Number of categorical columns: 17


In [ ]:
df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True
)

print("Categorical encoding completed.")
print("New dataset shape:", df.shape)

Categorical encoding completed.
New dataset shape: (1447, 4409)


In [ ]:
print("Final dataset shape:", df.shape)

print("Total missing values:",
      df.isnull().sum().sum())

print("Total duplicate rows:",
      df.duplicated().sum())

Final dataset shape: (1447, 4409)
Total missing values: 0
Total duplicate rows: 0


In [ ]:
df.head()

,patientunitstayid,patienthealthsystemstayid,hospitalid,wardid,admissionheight,hospitaladmitoffset,hospitaldischargeyear,hospitaldischargeoffset,unitvisitnumber,admissionweight,...,uniquepid_010-10570,uniquepid_010-10578,uniquepid_010-10636,uniquepid_010-10638,uniquepid_010-10674,uniquepid_010-10676,uniquepid_010-107,uniquepid_010-10712,uniquepid_010-10728,uniquepid_010-10734
0,-1.132469,-1.158683,-1.272693,-1.336145,0.925308,0.698450,0.731071,-0.500157,0.0,1.547460,...,False,False,False,False,False,False,False,False,False,False
1,-1.158384,-1.187406,-1.311036,-1.348736,0.906222,0.479388,-1.367857,-0.960001,0.0,1.096624,...,False,False,False,False,False,False,False,False,False,False
2,-1.103624,-1.126731,-1.311036,-1.348736,0.667650,0.704234,-1.367857,-1.000170,0.0,-0.178469,...,False,False,False,False,False,False,False,False,False,False
3,-1.103406,-1.126505,-1.311036,-1.348736,1.631483,0.104886,-1.367857,-0.957409,0.0,2.514025,...,False,False,False,False,False,False,False,False,False,False
4,-1.112176,-1.136176,-1.272693,-1.317259,0.419534,-1.404688,-1.367857,0.100605,0.0,-0.178469,...,False,False,False,False,False,False,False,False,False,False
